# Output-digit-balanced carry localisation

This is one final, predeclared test of whether the selected mathematics TopK-256 SAE contains a compact carry-associated panel that the single-prompt attribution graph missed. It does **not** retrain the model or SAE.

The experiment uses all 57,344 latents across layers 4, 8, 12, 16, 20, 24 and 28. Features are ranked only on discovery prompts by their carry-versus-no-carry activation difference **within groups that predict the same tens digit**. The Top-10 order is then frozen and tested on disjoint matched carry/no-carry pairs using final-token, error-preserving inhibition. A raw-MLP conditional direction is evaluated in parallel to distinguish an SAE-basis failure from a weak final-token MLP site.

The joint primary criterion is fixed before the run: the Top-10 panel must retain a positive output-digit-conditioned activation effect on confirmation data, reduce the carry-target gap, and have a carry-minus-control causal bootstrap interval wholly below zero. Secondary panel sizes and random controls must not be used to replace the primary result.

## 1. Mount Drive and fetch the current repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = '/content/test_run'

def run_cmd(command):
    print('$', ' '.join(map(str, command)))
    subprocess.run(command, check=True)

github_ok = False
try:
    checkout = repo_dir
    if os.path.isdir(os.path.join(checkout, '.git')):
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        if os.path.exists(checkout) and os.listdir(checkout):
            checkout = '/content/test_run_github'
        if os.path.isdir(os.path.join(checkout, '.git')):
            run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
        elif os.path.exists(checkout) and os.listdir(checkout):
            raise RuntimeError(f'{checkout} exists but is not a git repository')
        else:
            run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
    os.chdir(checkout)
    github_ok = True
    print('Using GitHub checkout:', os.getcwd())
except Exception as exc:
    print('GitHub checkout failed; using Drive project.zip backup.')
    print(repr(exc))

if not github_ok:
    zip_path = '/content/drive/MyDrive/mphil-project/project.zip'
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Could not find {zip_path}')
    run_cmd(['unzip', '-q', '-o', zip_path, '-d', '/content/'])
    for candidate in ['/content/test_run', '/content/mphil_project/test_run', '/content']:
        if os.path.isdir(os.path.join(candidate, 'src')) and os.path.isdir(os.path.join(candidate, 'configs')):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError('Could not locate the extracted repository root')

print('Current working directory:', os.getcwd())

## 2. Install dependencies and restore the selected SAE

In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" accelerate matplotlib pandas
!pip install -q -e .
!python data/generate_datasets.py --capitals

from pathlib import Path
required_files = [
    Path('src/math_carry_balanced_localization.py'),
    Path('configs/sae_math_topk256_config.yaml'),
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Push the balanced-localisation code to GitHub before running this notebook: ' +
        ', '.join(missing_files)
    )

In [ ]:
import json
import shutil
import sys

LAYERS = [4, 8, 12, 16, 20, 24, 28]
DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_CHECKPOINT = DRIVE_ROOT / 'mechanistic_data' / 'topk_math_retrain' / 'k256'
DRIVE_RETRAIN_OUTPUT = DRIVE_ROOT / 'outputs' / 'topk_math_retrain'
DRIVE_FOLLOWUP_OUTPUT = DRIVE_ROOT / 'outputs' / 'topk_math_followup'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs' / 'math_carry_localization'
LOCAL_CHECKPOINT = Path('mechanistic_data/sae_checkpoints_math_topk256')
LOCAL_OUTPUT = Path('outputs/math_carry_localization')

for path in (DRIVE_OUTPUT, LOCAL_CHECKPOINT, LOCAL_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

for layer in LAYERS:
    for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json'):
        source = DRIVE_CHECKPOINT / name
        destination = LOCAL_CHECKPOINT / name
        if not source.exists():
            raise FileNotFoundError(f'Missing selected TopK checkpoint artifact: {source}')
        if not destination.exists():
            shutil.copy2(source, destination)
            print('Restored', destination)

PRIOR_RESULTS = [
    DRIVE_RETRAIN_OUTPUT / 'math_topk256_heldout_specificity.json',
    DRIVE_FOLLOWUP_OUTPUT / 'math_topk256_carry_feature_screen.json',
]
missing_prior = [str(path) for path in PRIOR_RESULTS if not path.exists()]
if missing_prior:
    raise FileNotFoundError(
        'The prior result files are required so their cases can be excluded: ' +
        ', '.join(missing_prior)
    )

RESULT_PATH = DRIVE_OUTPUT / 'math_topk256_balanced_carry_localization.json'
CACHE_PATH = DRIVE_OUTPUT / 'math_topk256_balanced_carry_activations.npz'
print('Result checkpoint:', RESULT_PATH)
print('Activation cache:', CACHE_PATH)
print('Previously inspected result files:', *PRIOR_RESULTS, sep='\n  ')

## 3. Run the balanced discovery and frozen confirmation

The result JSON is checkpointed to Drive after every causal panel, and the activation matrix is cached after capture. If Colab disconnects, rerun this cell with the same arguments. Do not add `--overwrite` unless intentionally restarting the entire predeclared protocol.

In [ ]:
command = [
    sys.executable, '-m', 'src.math_carry_balanced_localization',
    '--sae-config', 'configs/sae_math_topk256_config.yaml',
    '--candidate-pairs', '149',
    '--discovery-pairs', '32',
    '--confirmation-pairs', '32',
    '--seed', '4787',
    '--batch-size', '4',
    '--panel-sizes', '1', '3', '5', '10', '20',
    '--primary-panel-size', '10',
    '--random-panels', '5',
    '--minimum-active-fraction', '0.10',
    '--minimum-positive-stratum-fraction', '0.60',
    '--exclude-json', *map(str, PRIOR_RESULTS),
    '--output', str(RESULT_PATH),
    '--activation-cache', str(CACHE_PATH),
]
run_cmd(command)

## 4. Inspect the predeclared result

In [ ]:
import pandas as pd
from IPython.display import display

result = json.loads(RESULT_PATH.read_text())
print('Status:', result['status'])
print('Excluded previously inspected pairs:', result['exclusions']['excluded_case_count'])
print('Eligible fresh pairs:', result['baseline_screen']['eligible_pairs'])
print('Discovery pairs:', len(result['split']['discovery_case_keys']))
print('Confirmation pairs:', len(result['split']['confirmation_case_keys']))
print('Joint predeclared success:', result['primary_result']['supports_compact_carry_selectivity_under_predeclared_rule'])
print(result['primary_result']['interpretation_gate'])

raw_table = pd.DataFrame([
    {
        'layer': row['layer'],
        'discovery conditioned AUC': row['discovery']['output_digit_conditioned_auc'],
        'confirmation conditioned AUC': row['confirmation']['output_digit_conditioned_auc'],
        'confirmation effect': row['confirmation']['mean_within_digit_carry_minus_no_carry'],
        'confirmation 95% CI': row['confirmation']['bootstrap_95_ci_mean_within_digit_difference'],
    }
    for row in result['raw_mlp_localisation']
]).set_index('layer')
display(raw_table)

ranking_table = pd.DataFrame(result['sae_feature_discovery']['top_200_ranking'][:20])
display(ranking_table[[
    'rank', 'key', 'conditional_standardised_effect',
    'positive_output_digit_strata', 'total_output_digit_strata',
    'carry_active_fraction', 'no_carry_active_fraction',
]])

In [ ]:
panel_table = pd.DataFrame([
    {
        'panel': panel['name'],
        'kind': panel['kind'],
        'features': panel['feature_count'],
        'activation confirmation': panel['activation_validation']['confirmation']['mean_within_digit_carry_minus_no_carry'],
        'activation CI': panel['activation_validation']['confirmation']['bootstrap_95_ci_mean_within_digit_difference'],
        'carry target delta': panel['causal_summary']['mean_carry_target_delta'],
        'no-carry control delta': panel['causal_summary']['mean_no_carry_control_delta'],
        'causal specificity': panel['causal_summary']['mean_paired_difference'],
        'causal 95% CI': panel['causal_summary']['bootstrap_95_ci_mean_paired_difference'],
    }
    for panel in result['causal_confirmation']['panels']
])
display(panel_table)
display(pd.Series(result['primary_result']['activation_confirmation']).to_frame('activation confirmation'))
display(pd.Series(result['primary_result']['causal_confirmation']).to_frame('causal confirmation'))

## 5. Save a diagnostic figure and local result copy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

raw_layers = [row['layer'] for row in result['raw_mlp_localisation']]
raw_auc = [row['confirmation']['output_digit_conditioned_auc'] for row in result['raw_mlp_localisation']]
axes[0].plot(raw_layers, raw_auc, marker='o', color='#2F6FA3')
axes[0].axhline(0.5, color='0.5', linestyle='--', linewidth=1)
axes[0].set(title='Raw MLP carry decodability', xlabel='Layer', ylabel='Conditioned confirmation AUC', ylim=(0.4, 1.0))

top = result['sae_feature_discovery']['top_200_ranking'][:10]
axes[1].barh([row['key'] for row in reversed(top)], [row['conditional_standardised_effect'] for row in reversed(top)], color='#7B5AA6')
axes[1].set(title='Frozen discovery ranking', xlabel='Within-digit standardised effect')

ranked_panels = [panel for panel in result['causal_confirmation']['panels'] if panel['kind'] == 'balanced_activation_ranked_prefix']
x = [panel['feature_count'] for panel in ranked_panels]
y = [panel['causal_summary']['mean_paired_difference'] for panel in ranked_panels]
lo = [panel['causal_summary']['bootstrap_95_ci_mean_paired_difference'][0] for panel in ranked_panels]
hi = [panel['causal_summary']['bootstrap_95_ci_mean_paired_difference'][1] for panel in ranked_panels]
axes[2].errorbar(x, y, yerr=[np.asarray(y) - np.asarray(lo), np.asarray(hi) - np.asarray(y)], marker='o', color='#C53B4C', capsize=3)
axes[2].axhline(0, color='0.3', linewidth=1)
axes[2].set(title='Frozen causal specificity', xlabel='Top-N features', ylabel='Carry minus no-carry gap delta')

fig.suptitle('Output-digit-balanced carry localisation', fontweight='bold')
fig.tight_layout()
figure_png = DRIVE_OUTPUT / 'fig_math_balanced_carry_localization.png'
figure_pdf = DRIVE_OUTPUT / 'fig_math_balanced_carry_localization.pdf'
fig.savefig(figure_png, dpi=220, bbox_inches='tight')
fig.savefig(figure_pdf, bbox_inches='tight')
plt.show()

for source in (RESULT_PATH, figure_png, figure_pdf):
    destination = LOCAL_OUTPUT / source.name
    shutil.copy2(source, destination)
    print('Copied', destination)

## Interpretation gate

- **Raw MLP confirmation AUC near 0.5:** carry is not linearly exposed at this final-token MLP site under output-digit control; another component or position would be a future study.
- **Raw MLP signal but no held-out SAE activation signal:** the existing SAE basis does not align cleanly with the decodable carry direction.
- **Held-out SAE activation signal but failed causal specificity:** carry is decodable from the panel, but the selected features are not established as a selective causal mechanism.
- **Joint primary rule passes:** report a compact carry-selective candidate only after one independent replication with a new seed and untouched cases.
- **Joint primary rule fails:** stop searching within this SAE. Do not replace the Top-10 result with a better-looking secondary panel.